# Suraksha - RAG Utilities for Fraud Explanations

**Utility functions for retrieving fraud explanations**

Shared Component - Used by both advanced and baseline solutions

## Purpose:
- Load pre-built FAISS index from rag_pipeline
- Search RBI guidelines by fraud type or query
- Generate fraud explanations grounded in regulatory context
- Provide actionable prevention measures

## Key Functions:
- `load_rag_index()` - Load FAISS index and metadata
- `search_rbi_guidelines(query, k=3)` - Search for relevant guideline chunks
- `get_explanation(fraud_type, transaction_details=None)` - Get comprehensive fraud explanation

## Usage:
```python
from rag_utils import load_rag_index, get_explanation

# Load RAG index once
load_rag_index()

# Get explanation for detected fraud
explanation = get_explanation("Velocity Fraud", {
    'sender_txn_count_1min': 7,
    'time_since_last_txn_sec': 30
})
print(explanation)
```

In [0]:
# Install dependencies if not already available
# Note: If you've just run rag_pipeline in this session, packages should already be installed
try:
    import sentence_transformers
    import faiss
    print("✓ Dependencies already installed")
except ImportError:
    print("📦 Installing RAG dependencies (one-time setup)...")
    %pip install sentence-transformers faiss-cpu --quiet
    print("✓ Packages installed")
    print("🔄 Restarting Python...")
    dbutils.library.restartPython()

In [0]:
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🚀 SURAKSHA - RAG UTILITIES")
print("="*70)

# Global variables (loaded once, reused)
_index = None
_chunks = None
_metadata = None
_model = None
_model_info = None

print("\n✓ RAG utilities imported")
print("✓ Ready to load index")

In [0]:
def load_rag_index():
    """
    Load the pre-built FAISS index and metadata from disk.
    Uses dynamic path based on current user for reproducibility.
    
    Returns:
        tuple: (index, chunks, metadata, model, model_info)
    """
    global _index, _chunks, _metadata, _model, _model_info
    
    # Check if already loaded
    if _index is not None:
        print("ℹ️ RAG index already loaded in memory")
        return _index, _chunks, _metadata, _model, _model_info
    
    print("🔄 Loading RAG index...")
    
    # Get current user dynamically (consistent with all other notebooks)
    try:
        username = spark.sql("SELECT current_user()").collect()[0][0]
    except:
        username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
    
    rag_dir = f"/Workspace/Users/{username}/Suraksha/rag/"
    index_dir = rag_dir + "faiss_index/"
    
    # Try Workspace path first, fallback to DBFS
    if not os.path.exists(index_dir + "index.faiss"):
        print("⚠️  Workspace path not found, trying DBFS...")
        dbfs_rag_dir = "/dbfs/suraksha/rag/"
        if os.path.exists(dbfs_rag_dir + "index.faiss"):
            index_dir = dbfs_rag_dir
            rag_dir = dbfs_rag_dir
            print("✓ Found index in DBFS fallback path")
        else:
            raise FileNotFoundError(
                "RAG index not found in Workspace or DBFS. "
                "Run rag_pipeline notebook first."
            )
    
    try:
        # Load FAISS index
        index_path = index_dir + "index.faiss"
        _index = faiss.read_index(index_path)
        print(f"✓ FAISS index loaded: {_index.ntotal} vectors")
        
        # Load text chunks
        chunks_path = rag_dir + "chunks.pkl"
        with open(chunks_path, 'rb') as f:
            _chunks = pickle.load(f)
        print(f"✓ Text chunks loaded: {len(_chunks)} chunks")
        
        # Load chunk metadata
        metadata_path = rag_dir + "metadata.pkl"
        with open(metadata_path, 'rb') as f:
            _metadata = pickle.load(f)
        print(f"✓ Chunk metadata loaded")
        
        # Load model info
        model_info_path = rag_dir + "model_info.pkl"
        with open(model_info_path, 'rb') as f:
            _model_info = pickle.load(f)
        print(f"✓ Model info loaded: {_model_info['model_name']}")
        
        # Load sentence transformer model
        _model = SentenceTransformer(_model_info['model_name'])
        print(f"✓ Embedding model loaded")
        
        print(f"\n✅ RAG index ready (user: {username})")
        return _index, _chunks, _metadata, _model, _model_info
        
    except FileNotFoundError as e:
        print(f"❌ Error: RAG index not found at {rag_dir}")
        print("Please run the rag_pipeline notebook first to build the index.")
        raise e

print("\n✓ load_rag_index() function defined")

In [0]:
def search_rbi_guidelines(query, k=3, fraud_type_filter=None):
    """
    Search RBI guidelines for relevant information.
    
    Args:
        query: Search query (natural language)
        k: Number of top results to return
        fraud_type_filter: Optional fraud type to filter results
    
    Returns:
        list: List of dicts with 'text', 'fraud_type', 'score'
    """
    global _index, _chunks, _metadata, _model
    
    # Check if index is loaded
    if _index is None:
        raise RuntimeError("RAG index not loaded. Call load_rag_index() first.")
    
    # Encode query
    query_embedding = _model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    
    # Search
    distances, indices = _index.search(query_embedding, k * 3 if fraud_type_filter else k)
    
    # Build results
    results = []
    for idx, score in zip(indices[0], distances[0]):
        fraud_type = _metadata[idx]
        
        # Apply fraud type filter if specified
        if fraud_type_filter and fraud_type != fraud_type_filter:
            continue
        
        results.append({
            'text': _chunks[idx],
            'fraud_type': fraud_type,
            'score': float(score)
        })
        
        # Stop if we have enough results
        if len(results) >= k:
            break
    
    return results

print("✓ search_rbi_guidelines() function defined")

In [0]:
def get_explanation(fraud_type, transaction_details=None, include_prevention=True):
    """
    Get comprehensive fraud explanation with RBI guidelines.
    
    Args:
        fraud_type: Name of fraud type (e.g., "Velocity Fraud")
        transaction_details: Optional dict with transaction features
        include_prevention: Whether to include prevention measures
    
    Returns:
        dict: Comprehensive explanation with guidelines, indicators, prevention
    """
    global _index, _chunks, _metadata, _model
    
    # Check if index is loaded
    if _index is None:
        raise RuntimeError("RAG index not loaded. Call load_rag_index() first.")
    
    # Search for relevant guideline chunks
    results = search_rbi_guidelines(
        query=f"{fraud_type} indicators and prevention",
        k=3,
        fraud_type_filter=fraud_type
    )
    
    # Build explanation
    explanation = {
        'fraud_type': fraud_type,
        'description': f"This transaction shows patterns consistent with {fraud_type}.",
        'rbi_guidelines': [],
        'key_indicators': [],
        'prevention_measures': [],
        'severity': 'High',
        'confidence': 'High',
        'hindi_description': None,
        'translation_ready': False
    }
    
    # Extract information from retrieved chunks
    for result in results:
        text = result['text']
        explanation['rbi_guidelines'].append(text[:400])  # First 400 chars
    
    # Static indicators based on fraud type (keyword extraction)
    fraud_indicators = {
        "Velocity Fraud": [
            "- Multiple transactions within 60 seconds",
            "- Transaction count > 5 in 1 minute",
            "- Rapid depletion of account balance"
        ],
        "Mule Account": [
            "- High inbound count followed by immediate outbound",
            "- Round amount transactions (₹5000, ₹10000)",
            "- Multiple senders to single receiver"
        ],
        "SIM Swap": [
            "- SIM changed within last 7 days",
            "- New device registered same day",
            "- Failed MPIN attempts before success"
        ],
        "Device Takeover": [
            "- Unrecognized device ID",
            "- Location changed from user's usual state",
            "- Amount 2.5x above user average"
        ],
        "Beneficiary Manipulation": [
            "- Transaction at odd hours (11 PM - 5 AM)",
            "- First-time sender-receiver pair",
            "- Round amount with MPIN attempts"
        ],
        "Amount Anomaly": [
            "- Amount z-score > 3 standard deviations",
            "- Transaction in top 5% of user history",
            "- Amount near UPI limit (₹50,000)"
        ],
        "Temporal Anomaly": [
            "- Transaction between 2 AM - 5 AM",
            "- Weekend large transaction",
            "- Outside user's typical transaction hours"
        ],
        "Merchant Fraud": [
            "- High-risk merchant category",
            "- Amount mismatch for merchant type",
            "- First transaction with this merchant"
        ],
        "Failed-Then-Success": [
            "- 2+ failed attempts before success",
            "- MPIN brute force pattern",
            "- Credential stuffing indicators"
        ]
    }
    
    fraud_prevention = {
        "Velocity Fraud": [
            "- Enable UPI transaction frequency limits",
            "- Set daily transaction count caps",
            "- Enable velocity alerts via SMS"
        ],
        "Mule Account": [
            "- Report to bank's fraud team immediately",
            "- File complaint on cybercrime.gov.in",
            "- Contact RBI CPFIR portal"
        ],
        "SIM Swap": [
            "- Contact your telecom provider immediately",
            "- Freeze banking transactions temporarily",
            "- Enable SIM change alerts with bank"
        ],
        "Device Takeover": [
            "- Change UPI PIN immediately",
            "- Deregister unknown devices in app",
            "- Enable device binding in bank settings"
        ],
        "Beneficiary Manipulation": [
            "- Verify receiver VPA before sending",
            "- Use UPI name verification feature",
            "- Never pay under phone/video pressure"
        ],
        "Amount Anomaly": [
            "- Set UPI transaction amount alerts",
            "- Enable 2FA for high-value transactions",
            "- Review and confirm large transfers"
        ],
        "Temporal Anomaly": [
            "- Set nighttime transaction blocks",
            "- Enable geolocation-based alerts",
            "- Review transaction at safe time"
        ],
        "Merchant Fraud": [
            "- Verify merchant VPA carefully",
            "- Check merchant category before payment",
            "- Use UPI merchant verification"
        ],
        "Failed-Then-Success": [
            "- Change MPIN after failed attempts",
            "- Enable login attempt notifications",
            "- Lock account after 3 failed attempts"
        ]
    }
    
    explanation['key_indicators'] = fraud_indicators.get(
        fraud_type, ["- Suspicious transaction pattern detected"]
    )
    explanation['prevention_measures'] = fraud_prevention.get(
        fraud_type, ["- Contact your bank immediately"]
    )
    
    # Add transaction-specific context if provided
    if transaction_details:
        explanation['transaction_context'] = _build_transaction_context(fraud_type, transaction_details)
    
    return explanation

def _build_transaction_context(fraud_type, details):
    """
    Build transaction-specific context based on fraud type and features.
    """
    context = []
    
    # Fraud-type specific context
    if fraud_type == "Velocity Fraud":
        if details.get('sender_txn_count_1min', 0) > 5:
            context.append(f"High transaction velocity: {details['sender_txn_count_1min']} transactions in 1 minute")
        if details.get('time_since_last_txn_sec', 999) < 60:
            context.append(f"Very short time between transactions: {details['time_since_last_txn_sec']} seconds")
    
    elif fraud_type == "SIM Swap":
        if details.get('sim_change_recent'):
            context.append("Recent SIM card change detected")
        if details.get('is_odd_hours'):
            context.append("Transaction during odd hours (11 PM - 6 AM)")
        if details.get('mpin_attempts', 0) > 0:
            context.append(f"Failed authentication attempts: {details['mpin_attempts']}")
    
    elif fraud_type == "Device Takeover":
        if details.get('device_changed_flag'):
            context.append("New device registration detected")
        if details.get('location_changed_flag'):
            context.append("Location change detected")
    
    elif fraud_type == "Mule Account":
        if details.get('receiver_inbound_count_1h', 0) > 5:
            context.append(f"High inbound transactions: {details['receiver_inbound_count_1h']} in 1 hour")
        if details.get('receiver_outbound_count_1h', 0) > 5:
            context.append(f"High outbound transactions: {details['receiver_outbound_count_1h']} in 1 hour")
    
    elif fraud_type == "Amount Anomaly":
        if details.get('amount_zscore', 0) > 2:
            context.append(f"Amount significantly above user's average (z-score: {details['amount_zscore']:.2f})")
        if details.get('amount_percentile', 50) > 90:
            context.append(f"Amount in top {100-details['amount_percentile']:.0f}% of user's transactions")
    
    return context if context else ["Transaction pattern matches known fraud indicators"]

print("✓ get_explanation() function defined")

In [0]:
print("\n📊 Demo: Testing RAG utilities...\n")

# Load the RAG index
index, chunks, metadata, model, model_info = load_rag_index()
print(f"\n✓ Index loaded successfully")
print(f"   Total vectors: {index.ntotal}")
print(f"   Total chunks: {len(chunks)}")
print(f"   Fraud types covered: {len(set(metadata))}\n")

# Example 1: Search for velocity fraud information
print("="*70)
print("🔍 Example 1: Searching for velocity fraud indicators")
print("="*70)

results = search_rbi_guidelines("What are the signs of velocity fraud?", k=2)
for i, result in enumerate(results, 1):
    print(f"\nResult {i} (Score: {result['score']:.4f}):")
    print(f"Fraud Type: {result['fraud_type']}")
    print(f"Text: {result['text'][:300]}...\n")

# Example 2: Get comprehensive explanation
print("="*70)
print("🔍 Example 2: Getting fraud explanation with transaction details")
print("="*70)

txn_details = {
    'sender_txn_count_1min': 7,
    'sender_txn_count_1hour': 25,
    'time_since_last_txn_sec': 30,
    'amount_inr': 5000
}

explanation = get_explanation("Velocity Fraud", transaction_details=txn_details)

print(f"\n🚨 Fraud Type: {explanation['fraud_type']}")
print(f"Severity: {explanation['severity']}")
print(f"Confidence: {explanation['confidence']}")

print(f"\n📋 Transaction Context:")
for ctx in explanation.get('transaction_context', []):
    print(f"   • {ctx}")

print(f"\n⚠️ Key Indicators ({len(explanation['key_indicators'])} found):")
for indicator in explanation['key_indicators'][:3]:
    print(f"   {indicator}")

print(f"\n🛡️ Prevention Measures ({len(explanation['prevention_measures'])} found):")
for measure in explanation['prevention_measures'][:3]:
    print(f"   {measure}")

print("\n" + "="*70)
print("✅ RAG utilities working successfully!")
print("="*70)

In [0]:
print("\n" + "="*70)
print("✅ RAG UTILITIES READY!")
print("="*70)

print("\n🛠️ Available Functions:")
print("\n1. load_rag_index()")
print("   - Loads FAISS index and metadata")
print("   - Call once at startup")
print("   - Returns status dict with index info")

print("\n2. search_rbi_guidelines(query, k=3, fraud_type_filter=None)")
print("   - Search for relevant guideline chunks")
print("   - query: Natural language search query")
print("   - k: Number of results to return")
print("   - fraud_type_filter: Optional filter by fraud type")
print("   - Returns: List of dicts with text, fraud_type, score")

print("\n3. get_explanation(fraud_type, transaction_details=None, include_prevention=True)")
print("   - Get comprehensive fraud explanation")
print("   - fraud_type: Name of detected fraud type")
print("   - transaction_details: Dict with transaction features")
print("   - include_prevention: Whether to include prevention measures")
print("   - Returns: Dict with guidelines, indicators, prevention, context")

print("\n🎯 Usage Example:")
print("""
from rag_utils import load_rag_index, get_explanation

# Load index once
load_rag_index()

# Get explanation for detected fraud
explanation = get_explanation(
    fraud_type="Velocity Fraud",
    transaction_details={
        'sender_txn_count_1min': 7,
        'time_since_last_txn_sec': 30
    }
)

print(explanation['description'])
for ctx in explanation['transaction_context']:
    print(f"  • {ctx}")
""")

print("\n📚 Integration Points:")
print("   • Model serving (06_model_serving): Add explanations to predictions")
print("   • Streamlit app: Display fraud explanations to users")
print("   • API endpoint: Include explanations in JSON response")
print("   • Alerting system: Enrich alerts with regulatory context")

print("\n" + "="*70)